# MarsLandformNet V3 Colab GPU Training\nRuntime -> Change runtime type -> GPU, then run cells top-to-bottom.

In [ ]:
from google.colab import drive\nfrom pathlib import Path\nimport os, subprocess\n\ndrive.mount('/content/drive')\nREPO = Path('/content/MarsLab')\nif not REPO.exists():\n    subprocess.check_call(['git', 'clone', 'https://github.com/csparkresearch/MarsLab.git', str(REPO)])\nelse:\n    subprocess.check_call(['git', '-C', str(REPO), 'pull'])\n\nos.environ['MARSLAB_ROOT'] = str(REPO)\nprint('MARSLAB_ROOT=', os.environ['MARSLAB_ROOT'])\n\n# Keep compatibility with scripts that still use absolute /disk1 path\nsubprocess.check_call(['mkdir', '-p', '/disk1/cspark'])\nif Path('/disk1/cspark/MarsLab').exists() or Path('/disk1/cspark/MarsLab').is_symlink():\n    subprocess.check_call(['rm', '-rf', '/disk1/cspark/MarsLab'])\nsubprocess.check_call(['ln', '-s', str(REPO), '/disk1/cspark/MarsLab'])

In [ ]:
%cd /content/MarsLab\n!pip install -q --upgrade pip\n!pip install -q numpy torch torchvision scikit-learn

In [ ]:
# Optional: regenerate tile labels from Levy polygons (skip if already committed)\n!python scripts/marslandform_v2/data/generate_tile_labels.py

In [ ]:
# Full GPU training (inline, works even if helper scripts are not in remote branch)\n!python - <<'PY'\nimport json\nimport numpy as np\nimport torch\nfrom pathlib import Path\nfrom scripts.marslandform_v2.config import get_config\nfrom scripts.marslandform_v2.models.tile_classifier import TileLabelDataset, TileClassifierTrainer\n\nROOT = Path('/content/MarsLab')\nwith open(ROOT/'Data/HiRISE/v3_output/tile_labels_v3.json') as f:\n    tile_labels = json.load(f)\nwith open(ROOT/'Data/HiRISE/v3_output/tile_splits_v3.json') as f:\n    splits = json.load(f)\n\nemb = np.load(ROOT/'Data/HiRISE/v2_output/embeddings_ssl/embeddings_by_image.npy', allow_pickle=True).item()\ncfg = get_config().tile_classifier\ncfg.epochs = 100\ncfg.batch_size = 512\ncfg.lr = 1e-4\ncfg.patience = 15\n\ntrain_ds = TileLabelDataset(tile_labels, splits['train'], None, cfg, True, emb)\nval_ds = TileLabelDataset(tile_labels, splits['val'], None, cfg, False, emb)\ntrainer = TileClassifierTrainer(cfg, train_ds, val_ds, device='cuda' if torch.cuda.is_available() else 'cpu')\nresult = trainer.train()\nprint('best_f1=', result['best_f1'], 'best_epoch=', result['best_epoch'])\nPY

In [ ]:
# Compute F1 on val/test (inline)\n!python - <<'PY'\nimport json\nimport numpy as np\nimport torch\nfrom pathlib import Path\nfrom sklearn.metrics import f1_score\nfrom scripts.marslandform_v2.config import get_config, V3_CLASSES\nfrom scripts.marslandform_v2.models.tile_classifier import TileLabelDataset, TileLandformClassifier\n\nROOT = Path('/content/MarsLab')\nwith open(ROOT/'Data/HiRISE/v3_output/tile_labels_v3.json') as f:\n    tile_labels = json.load(f)\nwith open(ROOT/'Data/HiRISE/v3_output/tile_splits_v3.json') as f:\n    splits = json.load(f)\nemb = np.load(ROOT/'Data/HiRISE/v2_output/embeddings_ssl/embeddings_by_image.npy', allow_pickle=True).item()\ncfg = get_config().tile_classifier\nckpt = torch.load(ROOT/'Data/HiRISE/v3_output/models/best_tile_classifier.pt', map_location='cpu')\nmodel = TileLandformClassifier(cfg)\nmodel.load_state_dict(ckpt['model_state_dict'])\nmodel.eval()\n\ndef evaluate(split_name):\n    ds = TileLabelDataset(tile_labels, splits[split_name], None, cfg, False, emb)\n    yt, yp = [], []\n    with torch.no_grad():\n        for i in range(len(ds)):\n            s = ds[i]\n            pred = int(model(s['embedding'].unsqueeze(0), s['mola'].unsqueeze(0)).argmax(dim=1).item())\n            yp.append(pred)\n            yt.append(int(s['label'].item()))\n    overall = f1_score(yt, yp, average='macro', zero_division=0)\n    idx = [i for i, y in enumerate(yt) if y < 3]\n    landform = f1_score([yt[i] for i in idx], [yp[i] for i in idx], average='macro', zero_division=0) if idx else 0.0\n    per = {}\n    for c, name in enumerate(V3_CLASSES):\n        per[name] = f1_score([1 if y==c else 0 for y in yt], [1 if p==c else 0 for p in yp], zero_division=0)\n    return {'samples': len(ds), 'overall_macro_f1': overall, 'landform_macro_f1': landform, 'per_class_f1': per}\n\nprint(json.dumps({'val': evaluate('val'), 'test': evaluate('test')}, indent=2))\nPY